# Procena stanovnika preko plocica (agregaciona loss)

Svako naselje je pokriveno disjunktnim 2.24km Sentinel plocicama. ResNet-18 daje broj stanovnika
po plocici (softplus, nenegativno); suma plocica jednog naselja je predikcija za to naselje.
Loss poredi log1p(sumu) sa log1p(popisa). Time slika i labela odgovaraju i velikim naseljima.
Podela po opstinama (GroupKFold). Pracenje kroz MLflow.

## Instalacija

In [ ]:
%pip install -q timm mlflow
try:
    dbutils.library.restartPython()
except NameError:
    pass

## Konfiguracija

In [ ]:
import os, glob, zipfile, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm, mlflow
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# Colab: "/content/tiles_data" (raspakuje tiles_upload.zip). Databricks: postavi na svoj Volume.
DATA_DIR = "/content/tiles_data"
if os.path.isdir("/content") and not os.path.isdir(DATA_DIR + "/tiles"):
    zipfile.ZipFile("/content/tiles_upload.zip").extractall(DATA_DIR)
TILES = DATA_DIR + "/tiles"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
NB, PX = 6, 224
EP_HEAD, EP_FT = 3, 40
NASELJA_PER_BATCH = 8           # batch = 8 naselja (promenljiv broj plocica)
LR_HEAD, LR_FT = 1e-3, 3e-4
NUM_WORKERS = 8
print("device:", DEVICE)

## Podaci i podela (po naseljima, grupisano po opstini)

In [ ]:
idx = pd.read_csv(DATA_DIR + "/tiles_index.csv")
idx["fpath"] = idx.path.map(lambda p: TILES + "/" + p)
tab = pd.read_parquet(DATA_DIR + "/naselje_table.parquet")[["naselje_maticni_broj", "opstina_maticni_broj"]]
nasel = (idx.groupby("naselje_maticni_broj")
         .agg(pop=("pop", "first"), n_tiles=("path", "size")).reset_index()
         .merge(tab, on="naselje_maticni_broj", how="left"))
print(f"plocica {len(idx)} | naselja {len(nasel)} | plocica/naselje med {int(nasel.n_tiles.median())} max {int(nasel.n_tiles.max())}")

splitter = GroupKFold(n_splits=min(5, nasel.opstina_maticni_broj.nunique()))
tr, va = next(splitter.split(nasel, groups=nasel.opstina_maticni_broj))
train_nasel = nasel.iloc[tr].reset_index(drop=True)
val_nasel = nasel.iloc[va].reset_index(drop=True)
tiles_by = {mb: g.fpath.tolist() for mb, g in idx.groupby("naselje_maticni_broj")}
print("trening naselja", len(train_nasel), "val naselja", len(val_nasel))

## Normalizacija i dataset

In [ ]:
svi_tr = [p for mb in train_nasel.naselje_maticni_broj for p in tiles_by[mb]]
uzorak = np.stack([np.load(p) for p in random.sample(svi_tr, min(400, len(svi_tr)))]).astype("float32")
MEAN = uzorak.mean((0, 2, 3), keepdims=True).astype("float32")
STD = uzorak.std((0, 2, 3), keepdims=True).astype("float32") + 1e-6

class NaseljaTiles(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        red = self.frame.iloc[i]
        paths = tiles_by[red.naselje_maticni_broj]
        xs = []
        for p in paths:
            x = (np.load(p).astype("float32") - MEAN[0]) / STD[0]
            if self.augment:
                if np.random.rand() < 0.5: x = x[:, :, ::-1]
                if np.random.rand() < 0.5: x = x[:, ::-1, :]
                x = np.rot90(x, np.random.randint(4), axes=(1, 2))
            xs.append(np.ascontiguousarray(x))
        return torch.from_numpy(np.stack(xs)), float(red["pop"])

def collate(batch):
    tiles = torch.cat([b[0] for b in batch], 0)                       # [sumT, 6, 224, 224]
    pops = torch.tensor([b[1] for b in batch], dtype=torch.float32)   # [B]
    nid = torch.cat([torch.full((b[0].shape[0],), i, dtype=torch.long) for i, b in enumerate(batch)])
    return tiles, nid, pops

def seed_worker(wid):
    s = SEED + wid; np.random.seed(s); random.seed(s)

gen = torch.Generator().manual_seed(SEED)
train_dl = DataLoader(NaseljaTiles(train_nasel, augment=True), batch_size=NASELJA_PER_BATCH, shuffle=True,
                      collate_fn=collate, generator=gen, num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True, worker_init_fn=seed_worker)
val_dl = DataLoader(NaseljaTiles(val_nasel), batch_size=NASELJA_PER_BATCH, collate_fn=collate,
                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

## Model i agregaciona loss

In [ ]:
model = timm.create_model("resnet18", pretrained=True, in_chans=NB, num_classes=1).to(DEVICE)
loss_fn = nn.HuberLoss()
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def prodji(loader, treniraj, optim=None, freeze_bn=False):
    model.train(treniraj)
    if freeze_bn:
        for m in model.modules():
            if isinstance(m, nn.BatchNorm2d): m.eval()
    tot, n, PRED, TRUE = 0.0, 0, [], []
    for tiles, nid, pops in loader:
        tiles, nid, pops = tiles.to(DEVICE, non_blocking=True), nid.to(DEVICE), pops.to(DEVICE)
        with torch.set_grad_enabled(treniraj), torch.autocast("cuda", enabled=USE_AMP):
            raw = model(tiles).squeeze(1)                       # [sumT]
        counts = F.softplus(raw.float())                        # fp32, nenegativno
        nc = torch.zeros(len(pops), device=DEVICE).scatter_add(0, nid, counts)  # suma po naselju (fp32)
        loss = loss_fn(torch.log1p(nc), torch.log1p(pops))
        if treniraj:
            optim.zero_grad(); scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
        tot += loss.item() * len(pops); n += len(pops)
        PRED.append(nc.detach().cpu().numpy()); TRUE.append(pops.detach().cpu().numpy())
    return tot / n, np.concatenate(PRED), np.concatenate(TRUE)

## Trening (sa MLflow pracenjem)

In [ ]:
mlflow.start_run()
mlflow.log_params({"pristup": "plocice_agregacija", "backbone": "resnet18", "bands": NB,
                   "naselja_po_batchu": NASELJA_PER_BATCH, "ep_head": EP_HEAD, "ep_ft": EP_FT,
                   "lr_head": LR_HEAD, "lr_ft": LR_FT, "seed": SEED, "n_naselja": len(nasel)})
istorija = []
best_r2, best_state = -1e9, None

def r2log(true, pred):
    return r2_score(np.log1p(true), np.log1p(np.clip(pred, 0, None)))

for naziv, param in model.named_parameters():
    param.requires_grad = naziv.startswith("fc")
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR_HEAD)
for e in range(EP_HEAD):
    tl, _, _ = prodji(train_dl, True, opt, freeze_bn=True)
    vl, P, Y = prodji(val_dl, False); r2 = r2log(Y, P); istorija.append((tl, vl, r2))
    mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=e)
    print(f"[glava {e}] train {tl:.3f} val {vl:.3f} R2 {r2:.3f}")

for param in model.parameters():
    param.requires_grad = True
opt = torch.optim.AdamW(model.parameters(), lr=LR_FT)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP_FT)
for e in range(EP_FT):
    tl, _, _ = prodji(train_dl, True, opt); sched.step()
    vl, P, Y = prodji(val_dl, False); r2 = r2log(Y, P); istorija.append((tl, vl, r2))
    mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=EP_HEAD + e)
    if r2 > best_r2:
        best_r2 = r2
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"[fine {e}] train {tl:.3f} val {vl:.3f} R2 {r2:.3f}")

model.load_state_dict(best_state)
mlflow.log_metric("best_val_r2", best_r2)
print(f"najbolji validacioni R2: {best_r2:.3f}")

## Evaluacija

In [ ]:
_, pred, stvarno = prodji(val_dl, False)
pred = np.clip(pred, 0, None)
metrike = {"val_r2_log": r2_score(np.log1p(stvarno), np.log1p(pred)),
           "val_mae_pop": mean_absolute_error(stvarno, pred),
           "val_rmse_pop": mean_squared_error(stvarno, pred) ** 0.5}
val2 = val_nasel.assign(pred=pred, stvarno=stvarno)
po_opstini = val2.groupby("opstina_maticni_broj")[["pred", "stvarno"]].sum()
metrike["agg_opstina_r2"] = r2_score(po_opstini.stvarno, po_opstini.pred) if len(po_opstini) > 1 else float("nan")
mlflow.log_metrics({k: float(v) for k, v in metrike.items()})
for k, v in metrike.items():
    print(f"{k}: {v:.3f}")

ep = range(len(istorija))
fig, ax = plt.subplots(2, 2, figsize=(12, 9))
ax[0, 0].plot(ep, [h[0] for h in istorija], label="trening"); ax[0, 0].plot(ep, [h[1] for h in istorija], label="validacija")
ax[0, 0].axvline(EP_HEAD - 0.5, ls=":", color="gray"); ax[0, 0].set_title("Huber gubitak po epohi"); ax[0, 0].set_xlabel("epoha"); ax[0, 0].legend()
ax[0, 1].plot(ep, [h[2] for h in istorija]); ax[0, 1].axhline(0, color="red", ls="--"); ax[0, 1].set_title("Validacioni R2 po epohi"); ax[0, 1].set_xlabel("epoha")
m = max(stvarno.max(), pred.max(), 1)
ax[1, 0].scatter(stvarno, pred, s=12, alpha=0.4); ax[1, 0].plot([1, m], [1, m], "r--")
ax[1, 0].set_xscale("log"); ax[1, 0].set_yscale("log"); ax[1, 0].set_title("Naselje: stvarno vs predvidjeno"); ax[1, 0].set_xlabel("stvarno"); ax[1, 0].set_ylabel("predvidjeno")
mm = max(po_opstini.stvarno.max(), po_opstini.pred.max(), 1)
ax[1, 1].scatter(po_opstini.stvarno, po_opstini.pred, s=35); ax[1, 1].plot([1, mm], [1, mm], "r--")
ax[1, 1].set_title("Agregacija po opstini (R2 %.2f)" % metrike["agg_opstina_r2"]); ax[1, 1].set_xlabel("stvarno"); ax[1, 1].set_ylabel("predvidjeno")
plt.tight_layout(); mlflow.log_figure(fig, "evaluacija.png"); plt.show()

mlflow.pytorch.log_model(model, name="model")
mlflow.end_run()